# Knowledge Distillation

A large model, or an ensemble of models, can be used during training because training does not have to run in real time.
A model deployed to many users must be small and fast. **Knowledge distillation** transfers what a large model has learned to a small model:
the small model learns not only the correct answers but also the probabilities that the large model assigns to every class.

This notebook follows the paper "Distilling the Knowledge in a Neural Network" by Hinton, Vinyals, and Dean (2015). It has eight sections:

| Section | Content |
|---|---|
| 1 | Load the data |
| 2 | Softmax with temperature: a worked example |
| 3 | The distillation loss and its gradient: a worked example, checked with automatic differentiation |
| 4 | Train the teacher |
| 5 | Student trained on hard labels only, compared with distillation |
| 6 | A smaller student: intermediate temperatures work best |
| 7 | A student that has never seen the digit 3 |
| 8 | Summary of results |

**How to run**: execute the cells from top to bottom. A GPU is faster ("Runtime → Change runtime type → T4 GPU");
on a CPU the whole notebook takes a few minutes.

## 1. Load the data

The data is a subset of the MNIST handwritten digits: 10,000 training images (1,000 of each digit 0 to 9) and the full test set of 10,000 images.
Each image is 28×28 grayscale with pixel values 0 to 255. The file is `kd_mnist.npz`, about 3.3 MB.

There are two ways to obtain the data; the next cell checks which one applies:

1. **Drag the file into Colab**: drag `kd_mnist.npz` into the "Files" panel on the left of Colab, so that it is in the default folder `/content`.
2. **Read it from the GitHub repository of this module**: if the file is not in the panel, the program downloads the same file automatically.

In [ ]:
import os
import urllib.request

import numpy as np

np.set_printoptions(suppress=True)  # print decimals without scientific notation, as on the slides

DATA = 'kd_mnist.npz'
URL = ('https://raw.githubusercontent.com/pychang-ai/115-1_DLMA/main/'
       'topics/knowledge-distillation/kd_mnist.npz')

if os.path.exists(DATA):
    print('Using the uploaded', DATA)
else:
    print(DATA, 'not found; downloading it from the GitHub repository')
    urllib.request.urlretrieve(URL, DATA)

d = np.load(DATA)
x_train = d['x_train'].reshape(-1, 784).astype('float32') / 255.0
y_train = d['y_train'].astype('int32')
x_test = d['x_test'].reshape(-1, 784).astype('float32') / 255.0
y_test = d['y_test'].astype('int32')
print('train', x_train.shape, 'test', x_test.shape)
print('training images per digit', np.bincount(y_train))
print('test images per digit    ', np.bincount(y_test))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for k, ax in enumerate(axes):
    i = np.where(y_train == k)[0][0]
    ax.imshow(x_train[i].reshape(28, 28), cmap='gray_r')
    ax.set_title(str(k))
    ax.axis('off')
plt.show()

## 2. Softmax with temperature

The numbers produced by the last layer of a neural network are called logits, written $z_i$. The softmax converts logits into probabilities. Equation (1) of the paper adds a temperature $T$:

$$q_i = \frac{\exp(z_i/T)}{\sum_j \exp(z_j/T)}$$

$T = 1$ gives the ordinary softmax. The larger $T$ is, the closer the class probabilities are to one another, and the "softer" the distribution.

**Worked example**: for a handwritten 2, the teacher's logits for the three classes "2, 3, 7" are $v = (8, 4, 0)$.
The computation has four steps: divide by $T$, take the exponential, sum, and divide each value by the sum.

In [ ]:
def softmax_T(z, T):
    """Equation (1): softmax at temperature T, returning every intermediate step."""
    z = np.asarray(z, dtype='float64')
    a = z / T
    e = np.exp(a)
    s = e.sum()
    return a, e, s, e / s


v = [8.0, 4.0, 0.0]
for T in [1, 4]:
    a, e, s, p = softmax_T(v, T)
    print(f'T = {T}')
    print('  step 1  z/T       ', np.round(a, 4))
    print('  step 2  exp(z/T)  ', np.round(e, 4))
    print('  step 3  sum       ', round(s, 4))
    print('  step 4  probability', np.round(p, 4))
    print(f'  ratio of P(3) to P(7): {p[1] / p[2]:.2f}')

At $T = 1$ the probabilities of "3" and "7" are 0.0180 and 0.0003, both close to 0; when the student is trained, these two numbers have almost no effect on the cross entropy.
At $T = 4$ they become 0.2447 and 0.0900, and the information "this 2 looks somewhat like a 3 and not much like a 7" now affects training.
The probabilities produced by the teacher at a high temperature are called **soft targets**; the correct answer itself (1 for class 2 and 0 elsewhere) is called the **hard target**.

In [ ]:
labels = ['2', '3', '7']
fig, axes = plt.subplots(1, 2, figsize=(8, 2.8), sharey=True)
for ax, T in zip(axes, [1, 4]):
    p = softmax_T(v, T)[3]
    ax.bar(labels, p, color=['#1F4E79', '#C03030', '#59636E'])
    for k, val in enumerate(p):
        ax.text(k, val + 0.02, f'{val:.4f}', ha='center')
    ax.set_title(f'teacher softmax, T = {T}')
    ax.set_ylim(0, 1.1)
plt.show()

## 3. The distillation loss and its gradient

The student's loss is a weighted sum of two terms:

- **Soft-target term**: the cross entropy $C_{soft} = -\sum_i p_i \log q_i$ between the teacher's probabilities $p$ at temperature $T$ and the student's probabilities $q$ at **the same temperature**.
- **Hard-target term**: the cross entropy $C_{hard}$ between the correct answer and the student's probabilities at $T = 1$.

$$L = \alpha \, T^2 \, C_{soft} + (1 - \alpha) \, C_{hard}$$

The paper points out that the gradients produced by the soft targets scale as $1/T^2$, so they are multiplied by $T^2$; this keeps the relative weight of the two terms unchanged when the temperature is changed.
The paper also reports that a considerably lower weight on the hard-target term generally works best. This notebook uses $\alpha = 0.9$.

The gradient of the soft-target term with respect to a student logit $z_i$ is equation (2) of the paper:

$$\frac{\partial C_{soft}}{\partial z_i} = \frac{1}{T}(q_i - p_i)$$

**Worked example**: the same handwritten 2, student logits $z = (4, 2, 2)$, temperature $T = 4$, $\alpha = 0.9$.

In [ ]:
T, alpha = 4.0, 0.9
z = np.array([4.0, 2.0, 2.0])
p = softmax_T(v, T)[3]          # teacher's soft targets
q = softmax_T(z, T)[3]          # student at T = 4
q1 = softmax_T(z, 1)[3]         # student at T = 1
y = np.array([1.0, 0.0, 0.0])   # the correct answer is 2

c_soft = -(p * np.log(q)).sum()
c_hard = -(y * np.log(q1)).sum()
loss = alpha * T ** 2 * c_soft + (1 - alpha) * c_hard
print('teacher soft targets p ', np.round(p, 4))
print('student q (T=4)        ', np.round(q, 4))
print('student q (T=1)        ', np.round(q1, 4))
print(f'C_soft = {c_soft:.4f}   C_hard = {c_hard:.4f}')
print(f'L = {alpha} × {T:g}² × {c_soft:.4f} + {1 - alpha:.1f} × {c_hard:.4f} = {loss:.4f}')

g_soft = (q - p) / T            # equation (2)
g_hard = q1 - y                 # gradient of the ordinary cross entropy
g = alpha * T ** 2 * g_soft + (1 - alpha) * g_hard
print()
print('soft-target gradient (q - p)/T, eq. (2)', np.round(g_soft, 4))
print('hard-target gradient q1 - y            ', np.round(g_hard, 4))
print('total gradient                         ', np.round(g, 4))

Compute the same quantities again with TensorFlow's automatic differentiation; the numbers must agree with the formulas above:

In [ ]:
import tensorflow as tf

zt = tf.Variable(z)
with tf.GradientTape() as tape:
    qt = tf.nn.log_softmax(zt / T)
    q1t = tf.nn.log_softmax(zt)
    L = alpha * T ** 2 * -tf.reduce_sum(p * qt) + (1 - alpha) * -tf.reduce_sum(y * q1t)
print('loss from automatic differentiation    ', round(float(L), 4))
print('gradient from automatic differentiation', np.round(tape.gradient(L, zt).numpy(), 4))

**Why multiply by $T^2$**: for the same logits, the higher the temperature, the smaller the gradient in equation (2). The table below compares $T = 1, 4, 20$:

In [ ]:
for t in [1, 4, 20]:
    gs = (softmax_T(z, t)[3] - softmax_T(v, t)[3]) / t
    print(f'T = {t:>2}  largest |gradient| {np.abs(gs).max():.6f}   after multiplying by T² {np.abs(gs).max() * t * t:.4f}')

When $T$ rises from 4 to 20, the gradient shrinks by a factor of about 20, close to $(20/4)^2 = 25$; after multiplying by $T^2$ the values are 0.85 and 1.05, of similar size.
The $1/T^2$ scaling is a high-temperature approximation; $T = 1$ is outside its range.

**At high temperature, distillation matches logits**: subtract the mean from the teacher's and the student's logits separately (the paper calls this zero-meaned).
When the temperature is high compared with the logits, equation (2) is approximately equation (4) of the paper:

$$\frac{\partial C_{soft}}{\partial z_i} \approx \frac{1}{N T^2}(z_i - v_i)$$

$N$ is the number of classes. This is equivalent to minimizing $\frac{1}{2}(z_i - v_i)^2$, that is, making the student's logits match the teacher's logits directly.

In [ ]:
N, T = 3, 20.0
zc, vc = z - z.mean(), np.array(v) - np.mean(v)
exact = (softmax_T(zc, T)[3] - softmax_T(vc, T)[3]) / T
approx = (zc - vc) / (N * T ** 2)
print('after subtracting the mean  z =', np.round(zc, 4), '  v =', np.round(vc, 4))
print('equation (2), exact      ', np.round(exact, 5))
print('equation (4), approximate', np.round(approx, 5))

## 4. Train the teacher

The teacher is a fully connected network with two hidden layers of 1,200 units each. As in the paper, it uses two methods against overfitting:
Dropout, and random shifts of the input image by up to 2 pixels. The output layer has no softmax and produces logits; the temperature is applied separately later.

In [ ]:
import keras
from keras import layers

keras.utils.set_random_seed(1)


def mlp(units, regularize=False, name='mlp'):
    """Fully connected network with two hidden layers and 10 output logits. regularize=True adds shifts and Dropout."""
    inp = keras.Input((784,))
    x = inp
    if regularize:
        x = layers.Reshape((28, 28, 1))(x)
        x = layers.RandomTranslation(2 / 28, 2 / 28, fill_mode='constant')(x)
        x = layers.Flatten()(x)
        x = layers.Dropout(0.2)(x)
    for _ in range(2):
        x = layers.Dense(units, activation='relu')(x)
        if regularize:
            x = layers.Dropout(0.5)(x)
    out = layers.Dense(10)(x)
    return keras.Model(inp, out, name=name)


def test_errors(model, bias=None):
    """Return the number of test errors and the logits. bias is added to the logits of each class."""
    logits = model.predict(x_test, batch_size=2000, verbose=0)
    if bias is not None:
        logits = logits + bias
    return int((logits.argmax(1) != y_test).sum()), logits


teacher = mlp(1200, regularize=True, name='teacher')
teacher.compile(keras.optimizers.Adam(1e-3),
                keras.losses.SparseCategoricalCrossentropy(from_logits=True))
teacher.fit(x_train, y_train, batch_size=128, epochs=30, verbose=0)
teacher_err, _ = test_errors(teacher)
print(f'teacher: {teacher.count_params():,} parameters, {teacher_err} errors on the 10,000 test images')

# teacher logits on the transfer set (the training set); every student uses this copy
v_train = teacher.predict(x_train, batch_size=2000, verbose=0)

## 5. Student trained on hard labels only, compared with distillation

The student is a small network with two hidden layers of 100 units and no regularization. The same student architecture is trained several times:

- **Hard labels only**: $\alpha = 0$, only the hard-target term.
- **Distillation**: $\alpha = 0.9$ with temperatures 1, 2, 4, 8, and 20.

The training loop is written by hand from the loss in Section 3. Every run uses the same initial weights and the same data order; only the loss differs.

In [ ]:
def train_student(units, T, alpha, mask=None, epochs=30, seed=2):
    """Train a student with L = α T² C_soft + (1 − α) C_hard. mask removes part of the transfer set."""
    keras.utils.set_random_seed(seed)
    student = mlp(units, name='student')
    opt = keras.optimizers.Adam(1e-3)
    x, yy, vv = (x_train, y_train, v_train) if mask is None else \
        (x_train[mask], y_train[mask], v_train[mask])
    ds = tf.data.Dataset.from_tensor_slices((x, yy, vv)).shuffle(len(x), seed=seed).batch(128)

    @tf.function
    def step(xb, yb, vb):
        with tf.GradientTape() as tape:
            zb = student(xb, training=True)
            pb = tf.nn.softmax(vb / T)                                     # teacher's soft targets
            c_soft = -tf.reduce_mean(tf.reduce_sum(pb * tf.nn.log_softmax(zb / T), 1))
            c_hard = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
                labels=yb, logits=zb))
            loss = alpha * T * T * c_soft + (1 - alpha) * c_hard
        grads = tape.gradient(loss, student.trainable_variables)
        opt.apply_gradients(zip(grads, student.trainable_variables))

    for _ in range(epochs):
        for xb, yb, vb in ds:
            step(xb, yb, vb)
    return student


TEMPS = [1, 2, 4, 8, 20]
results = {}
s = train_student(100, 1.0, 0.0)
results[(100, 'hard')] = test_errors(s)[0]
print(f'student 100, hard labels only: {results[(100, "hard")]} errors')
for t in TEMPS:
    s = train_student(100, float(t), 0.9)
    results[(100, t)] = test_errors(s)[0]
    print(f'student 100, distilled at T = {t:>2}: {results[(100, t)]} errors')

## 6. A smaller student: intermediate temperatures work best

The paper reports that when the student has 300 or more units per hidden layer, all temperatures above 8 give similar results;
when the student is reduced to 30 units per layer, temperatures from 2.5 to 4 work significantly better than higher or lower ones.
The reason is that the higher the temperature, the closer distillation is to the logit matching of Section 3: the student must also match the teacher's very negative logits,
which a very small student cannot do. The cell below repeats the experiment with a student of 30 units per layer:

In [ ]:
s = train_student(30, 1.0, 0.0)
results[(30, 'hard')] = test_errors(s)[0]
print(f'student 30, hard labels only: {results[(30, "hard")]} errors')
for t in TEMPS:
    s = train_student(30, float(t), 0.9)
    results[(30, t)] = test_errors(s)[0]
    print(f'student 30, distilled at T = {t:>2}: {results[(30, t)]} errors')

## 7. A student that has never seen the digit 3

The paper removes every 3 from the transfer set, so the student never sees a 3 during training.
A student trained on hard labels only cannot answer 3; a distilled student can still learn from the teacher's soft targets information such as "this 8 looks a little like a 3".

The paper also notes that most of the misclassified 3s are caused by a learned bias for class 3 that is too low: during training, no image has the answer 3.
After a constant is added to the logit of class 3, most 3s are classified correctly. The paper chooses this constant using the test set; this notebook does the same, so the resulting number is optimistic.

In [ ]:
mask = y_train != 3
print('images in the transfer set:', int(mask.sum()), '; of which digit 3:', int((y_train[mask] == 3).sum()))


def threes_wrong(logits):
    return int(((logits.argmax(1) != y_test) & (y_test == 3)).sum())


for name, t, a in [('hard labels only', 1.0, 0.0), ('distilled at T = 8', 8.0, 0.9)]:
    s = train_student(100, t, a, mask=mask)
    err, logits = test_errors(s)
    best = None
    for b in np.arange(0, 15.01, 0.25):
        shift = np.zeros(10, 'float32')
        shift[3] = b
        e = int(((logits + shift).argmax(1) != y_test).sum())
        if best is None or e < best[0]:
            best = (e, b, threes_wrong(logits + shift))
    results[('no3', name)] = (err, threes_wrong(logits), best)
    print(f'{name}: {err} errors, of which {threes_wrong(logits)} / 1010 are 3s')
    print(f'    after adding {best[1]:.2f} to the logit of 3: {best[0]} errors, of which {best[2]} / 1010 are 3s'
          f' ({100 * (1 - best[2] / 1010):.1f}% of 3s correct)')

## 8. Summary of results

The data and the models in this notebook are smaller than in the paper, so the numbers of errors differ from the paper; what matters is whether the three trends agree.

In [ ]:
print(f'{"model":<34}{"test errors":>12}')
print(f'{"teacher 1200-1200":<34}{teacher_err:>12}')
for units in [100, 30]:
    print(f'{"student %d, hard labels only" % units:<34}{results[(units, "hard")]:>12}')
    best_t = min(TEMPS, key=lambda t: results[(units, t)])
    print(f'{"student %d, distilled, best T=%d" % (units, best_t):<34}{results[(units, best_t)]:>12}')

fig, ax = plt.subplots(figsize=(7, 3.2))
for units, color in [(100, '#1F4E79'), (30, '#C03030')]:
    ax.plot(TEMPS, [results[(units, t)] for t in TEMPS], 'o-', color=color,
            label=f'student {units}-{units}, distilled')
    ax.axhline(results[(units, 'hard')], color=color, ls='--', label=f'student {units}-{units}, hard labels only')
ax.set_xscale('log')
ax.set_xticks(TEMPS)
ax.set_xticklabels([str(t) for t in TEMPS])
ax.set_xlabel('temperature T')
ax.set_ylabel('test errors (of 10,000)')
ax.legend(fontsize=8)
plt.show()

| Conclusion of the paper | Result in this notebook |
|---|---|
| A distilled student makes fewer errors than a student trained on hard labels only | Section 5 |
| When the student is very small, intermediate temperatures work best | Section 6 |
| A student that has never seen a 3 recognizes most 3s after the bias of class 3 is adjusted | Section 7 |

**Exercises**

1. In Section 3, change the student logits to $(8, 4, 0)$, identical to the teacher's. What is the soft-target gradient, and why?
2. In Section 5, change $\alpha$ to 0.5. How does the effect of distillation change?
3. In Section 7, remove the digit 7 instead and repeat the experiment.

---

Source: G. Hinton, O. Vinyals, J. Dean, "Distilling the Knowledge in a Neural Network," arXiv:1503.02531, 2015.
Data: MNIST handwritten digits (Y. LeCun, C. Cortes, C. J. C. Burges); this module uses 10,000 training images and the full test set.